In [0]:
from pyspark.sql import functions as F

tables = [
    "bronze_studies",
    "silver_studies",
    "silver_conditions",
    "silver_interventions",
    "silver_outcomes",
    "silver_locations",
    "gold_yearly_summary",
    "gold_country_summary",
    "gold_condition_trends",
    "serving_study_search",
    "serving_study_detail",
]

results = []

for table in tables:
    full_name = f"clinical_trials.{table}"

    count = (
        spark.sql(
            f"SELECT COUNT(*) AS cnt FROM {full_name}"
        )
        .first()["cnt"]
    )

    results.append((table, count))

metrics_df = spark.createDataFrame(
    results,
    ["table_name", "row_count"]
)

display(metrics_df)

table_name,row_count
bronze_studies,604566
silver_studies,604566
silver_conditions,1087211
silver_interventions,1022647
silver_outcomes,3567302
silver_locations,3532550
gold_yearly_summary,2306
gold_country_summary,4712
gold_condition_trends,378261
serving_study_search,604566


In [0]:
display(
    spark.read.table(
        "clinical_trials.pipeline_watermarks"
    )
)

pipeline_name,watermark_date,last_run_id,updated_at
clinical_trials_incremental,2026-09-25,b2383b87-bc94-4fcc-9d13-46aa3d56f0ba,2026-09-26T12:41:44.774Z


In [0]:
quality_df = spark.read.table(
    "clinical_trials.quality_results"
)

latest_run = (
    quality_df
    .orderBy(F.desc("executed_at"))
    .select("run_id")
    .first()["run_id"]
)

display(
    quality_df
    .filter(F.col("run_id") == latest_run)
    .orderBy("table_name", "rule_name")
)

checked_rows,executed_at,failed_rows,failure_rate,rule_name,run_id,table_name
1087211,2026-09-26T12:43:29.123Z,0,0.0,nct_id_exists_in_silver_studies,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_conditions
1022647,2026-09-26T12:43:31.020Z,0,0.0,nct_id_exists_in_silver_studies,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_interventions
3532550,2026-09-26T12:43:35.699Z,0,0.0,latitude_valid_range,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_locations
3532550,2026-09-26T12:43:36.573Z,0,0.0,longitude_valid_range,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_locations
3532550,2026-09-26T12:43:34.585Z,0,0.0,nct_id_exists_in_silver_studies,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_locations
3567302,2026-09-26T12:43:32.849Z,0,0.0,nct_id_exists_in_silver_studies,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_outcomes
604566,2026-09-26T12:43:27.578Z,0,0.0,completion_date_after_start_date,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_studies
604566,2026-09-26T12:43:26.051Z,0,0.0,enrollment_non_negative,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_studies
604566,2026-09-26T12:43:23.907Z,0,0.0,nct_id_not_null,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_studies
604566,2026-09-26T12:43:25.041Z,0,0.0,nct_id_unique,ec10dd32-d798-4b36-8d37-8a2a1e0ea076,silver_studies
